# CNN-EXP-048 — Ultra-Tail Risk Refinement (exact EXP047-source version)

This notebook is rebuilt against the **actual** `run_cnn_exp047_paired_risk_loss.py` source.

It fixes the failed generic bootstrap. EXP047 defines its strata by **rank cut counts**
(`1% / 5% / 15%`), not by literal quantile arrays, so EXP048 now changes the exact implementation:

- rank cuts: `0.1% / 1% / 5% / 15%`;
- five negative strata;
- target negative mixture: `0.065 / 0.065 / 0.145 / 0.175 / 0.55`;
- multiplier per stratum = `target_mass / realized_natural_fraction`;
- one candidate branch from EXP045 @50k;
- historical EXP047-B is the matched control;
- soft `0.90` GPU duty-cycle during training (not a hard hardware utilization cap).


In [1]:
from pathlib import Path
import hashlib, os, subprocess, sys, json

EXPECTED_PYTHON = Path(r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe").resolve()
assert Path(sys.executable).resolve() == EXPECTED_PYTHON, (sys.executable, EXPECTED_PYTHON)

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = next(
    c for c in (CURRENT, *CURRENT.parents)
    if (c / "README.md").is_file() and (c / "src/ml_project").is_dir()
)

AGENTS = next((p for p in (PROJECT_ROOT/"AGENTS.md", PROJECT_ROOT.parent/"AGENTS.md") if p.is_file()), None)
assert AGENTS is not None, "AGENTS.md not found"
print("AGENTS.md:", AGENTS)
print(AGENTS.read_text(encoding="utf-8"))

LOCAL_EXP047 = PROJECT_ROOT / "scripts/run_cnn_exp047_paired_risk_loss.py"
assert LOCAL_EXP047.is_file(), LOCAL_EXP047
EXPECTED_EXP047_SHA = "89b2bffd6da0ca76d785475a9379e12b2c06b2f8eecaac436b73dce048260d27"
actual_sha = hashlib.sha256(LOCAL_EXP047.read_bytes()).hexdigest()
assert actual_sha == EXPECTED_EXP047_SHA, (
    "Local EXP047 runner differs from the uploaded source. "
    f"actual={actual_sha} expected={EXPECTED_EXP047_SHA}"
)

TARGET = PROJECT_ROOT / "scripts/run_cnn_exp048_ultra_tail.py"
RUNNER_SOURCE = '"""CNN-EXP-048: WIDTH512 ultra-tail risk refinement from EXP045 @50k.\n\nThe command is intentionally process-oriented.  The notebook launches each long\nDirectML stage as a fresh process so a backend crash cannot destroy already\ncommitted checkpoints or the other branch.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport atexit\nimport copy\nfrom datetime import date\nfrom functools import lru_cache\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport sys\nimport time\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFont\nimport torch\n\n\nEXPERIMENT_ID = "CNN-EXP-048"\nEXPERIMENT_SLUG = "CNN_EXP048_WIDTH512_ULTRA_TAIL_RISK_REFINEMENT"\nBASE_EXPERIMENT = "CNN-EXP-045"\nBASE_STEP = 50_000\nCONTINUATION_STEPS = 20_000\nBATCH_SIZE = 4\nLOG_EVERY = 250\nLATEST_EVERY = 500\nSNAPSHOT_STEPS = {2_000, 5_000, 10_000, 15_000, 20_000}\nWARMUP_STEPS = 500\nSTART_LR = 3e-5\nPEAK_LR = 1e-4\nEND_LR = 1e-5\nINTENSITY_LOSS_WEIGHT = 0.1\nNEGATIVE_RANK_CUT_FRACTIONS = np.asarray([0.001, 0.01, 0.05, 0.15], dtype=np.float64)\nNEGATIVE_STRATA_NAMES = ["top_0_0p1pct", "top_0p1_1pct", "top_1_5pct", "top_5_15pct", "bottom_85pct"]\nTARGET_NEGATIVE_MIXTURE = np.asarray([0.065, 0.065, 0.145, 0.175, 0.55], dtype=np.float64)\nGPU_DUTY_CYCLE = float(os.environ.get("EPIC_GPU_DUTY_CYCLE", "0.90"))\nif not (0.50 <= GPU_DUTY_CYCLE <= 1.0):\n    raise RuntimeError(f"EPIC_GPU_DUTY_CYCLE must be in [0.50,1.0], got {GPU_DUTY_CYCLE}")\nSCORE_LEVELS = 100_001\nKEY_SHIFT = np.uint64(56)\nREGION_MASK = (np.uint64(1) << KEY_SHIFT) - np.uint64(1)\nBIT_SHIFTS = np.arange(32, dtype=np.uint32)\nEXPECTED_CHANNELS = 512\nEXPECTED_RF = 1029\nEXPECTED_PARAMETERS = 14_708_738\n\n\ndef project_root() -> Path:\n    here = Path(__file__).resolve()\n    for candidate in (here.parent, *here.parents):\n        if (candidate / "README.md").is_file() and (candidate / "src/ml_project").is_dir():\n            return candidate\n    raise RuntimeError("Project root was not found")\n\n\nROOT = project_root()\nSRC = ROOT / "src"\nif str(SRC) not in sys.path:\n    sys.path.insert(0, str(SRC))\n\nfrom ml_project.epic_baseline import load_baseline_sequences\nfrom ml_project.epic_cnn import (\n    DirectMLAdam,\n    ProfileWindowConfig,\n    ProfileWindowGenerator,\n    create_cnn_presence_intensity,\n    dml_weighted_huber,\n    dml_weighted_profile_bce_with_logits,\n    evaluate_presence_intensity_model,\n    forward_both_strands_multitask,\n)\nfrom ml_project.epic_data import load_split\n\n\nRUN_DIR = ROOT / "artifacts" / "experiments" / EXPERIMENT_SLUG / "run_001"\nBASE_RUN = (\n    ROOT\n    / "artifacts"\n    / "experiments"\n    / "CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029"\n    / "run_001"\n)\nBASE_CHECKPOINT = BASE_RUN / "model_step_50000.pt"\nBASE_METRICS = BASE_RUN / "validation_metrics_report.csv"\nEXP046_RUN = (\n    ROOT\n    / "artifacts"\n    / "experiments"\n    / "CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32"\n    / "run_001"\n)\nCV_SHARDS = EXP046_RUN / "_cv_eval_v3"\nORACLE_RUN = (\n    ROOT\n    / "artifacts"\n    / "experiments"\n    / "CNN_EXP046_REGIONAL_ACTIVITY_GATE"\n    / "run_001"\n)\nRISK_CACHE_DIR = RUN_DIR / "risk_strata_cache"\nRISK_CACHE = RISK_CACHE_DIR / "train_negative_stratum_u8.dat"\nRISK_MANIFEST = RISK_CACHE_DIR / "risk_strata_manifest.json"\nPREREG_PATH = RUN_DIR / "EXP048_preregistration.json"\nRESULT_PATH = RUN_DIR / "EXP048_RESULT.json"\nEXP047_RUN = ROOT / "artifacts" / "experiments" / "CNN_EXP047_WIDTH512_PAIRED_RISK_LOSS_CONTINUATION" / "run_001"\nEXP047_B_METRICS = EXP047_RUN / "branch_B" / "validation_metrics_report.csv"\nEXP047_B_STRONG = EXP047_RUN / "branch_B" / "strong_validation_metrics.csv"\nEXP047_B_COHORT = EXP047_RUN / "branch_B" / "fixed_cohort_scores.csv"\nNOTEBOOK_REL = (\n    "notebooks/experiments/CNN_experiments/"\n    "CNN_EXP048_ULTRA_TAIL_RISK_REFINEMENT.ipynb"\n)\n\n\ndef json_ready(value):\n    if isinstance(value, dict):\n        return {str(k): json_ready(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_ready(v) for v in value]\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating,)):\n        return float(value)\n    if isinstance(value, Path):\n        return str(value)\n    return value\n\n\ndef atomic_json(path: Path, payload: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    with open(tmp, "w", encoding="utf-8", newline="\\n") as handle:\n        json.dump(json_ready(payload), handle, ensure_ascii=False, indent=2, allow_nan=False)\n        handle.write("\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(tmp, path)\n\n\ndef atomic_csv(path: Path, frame: pd.DataFrame) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    frame.to_csv(tmp, index=False)\n    with open(tmp, "r+b") as handle:\n        os.fsync(handle.fileno())\n    os.replace(tmp, path)\n\n\ndef atomic_torch_save(path: Path, payload: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    with open(tmp, "wb") as handle:\n        torch.save(payload, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(tmp, path)\n\n\n@lru_cache(maxsize=32)\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef model_state_on_cpu(model: torch.nn.Module) -> dict:\n    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n\n\ndef tree_to_cpu(value):\n    if torch.is_tensor(value):\n        return value.detach().cpu().clone()\n    if isinstance(value, dict):\n        return {key: tree_to_cpu(item) for key, item in value.items()}\n    if isinstance(value, list):\n        return [tree_to_cpu(item) for item in value]\n    if isinstance(value, tuple):\n        return tuple(tree_to_cpu(item) for item in value)\n    return value\n\n\ndef planned_lr(local_step: int) -> float:\n    if not 1 <= local_step <= CONTINUATION_STEPS:\n        raise ValueError(f"local_step outside 1..{CONTINUATION_STEPS}: {local_step}")\n    if local_step <= WARMUP_STEPS:\n        fraction = local_step / WARMUP_STEPS\n        return START_LR + fraction * (PEAK_LR - START_LR)\n    fraction = (local_step - WARMUP_STEPS) / (CONTINUATION_STEPS - WARMUP_STEPS)\n    return END_LR + 0.5 * (PEAK_LR - END_LR) * (1.0 + math.cos(math.pi * fraction))\n\n\ndef soft_gpu_duty_pause(active_seconds: float) -> None:\n    """Soft throughput throttle; not a hard hardware-utilization cap."""\n    if GPU_DUTY_CYCLE >= 0.999999:\n        return\n    active_seconds = max(float(active_seconds), 0.0)\n    time.sleep(active_seconds * (1.0 / GPU_DUTY_CYCLE - 1.0))\n\n\ndef sigmoid_numpy(logits: np.ndarray) -> np.ndarray:\n    logits = np.asarray(logits, dtype=np.float64)\n    output = np.empty_like(logits)\n    positive = logits >= 0\n    output[positive] = 1.0 / (1.0 + np.exp(-logits[positive]))\n    exp_value = np.exp(logits[~positive])\n    output[~positive] = exp_value / (1.0 + exp_value)\n    return output\n\n\ndef score_bins(logits: np.ndarray) -> np.ndarray:\n    return np.rint(np.clip(sigmoid_numpy(logits), 0.0, 1.0) * 100_000).astype(np.int32)\n\n\ndef unpack_mask(mask_u32: np.ndarray) -> np.ndarray:\n    values = np.asarray(mask_u32, dtype=np.uint32)\n    return ((values[:, None] >> BIT_SHIFTS[None, :]) & np.uint32(1)).astype(bool)\n\n\ndef branch_dir(branch: str) -> Path:\n    if branch not in {"A", "B"}:\n        raise ValueError("branch must be A or B")\n    return RUN_DIR / f"branch_{branch}"\n\n\ndef _pid_is_running(pid: int) -> bool:\n    if pid <= 0:\n        return False\n    try:\n        os.kill(pid, 0)\n    except OSError:\n        return False\n    return True\n\n\ndef acquire_process_lock(path: Path, stage: str) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = {"pid": os.getpid(), "stage": stage, "created": time.time()}\n    for _ in range(2):\n        try:\n            descriptor = os.open(path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)\n        except FileExistsError:\n            try:\n                existing = json.loads(path.read_text(encoding="utf-8"))\n                existing_pid = int(existing.get("pid", -1))\n            except Exception:\n                existing_pid = -1\n                existing = {"unreadable": True}\n            if _pid_is_running(existing_pid):\n                raise RuntimeError(\n                    f"Stage already has a live process lock: {path}; details={existing}"\n                )\n            path.unlink(missing_ok=True)\n            continue\n        else:\n            with os.fdopen(descriptor, "w", encoding="utf-8") as handle:\n                json.dump(payload, handle)\n                handle.flush()\n                os.fsync(handle.fileno())\n            break\n    else:\n        raise RuntimeError(f"Could not acquire process lock: {path}")\n\n    def release() -> None:\n        try:\n            current = json.loads(path.read_text(encoding="utf-8"))\n            if int(current.get("pid", -1)) == os.getpid():\n                path.unlink(missing_ok=True)\n        except FileNotFoundError:\n            pass\n\n    atexit.register(release)\n\n\ndef preregister() -> None:\n    RUN_DIR.mkdir(parents=True, exist_ok=True)\n    payload = {\n        "experiment_id": EXPERIMENT_ID,\n        "created": date.today().isoformat(),\n        "status": "pre_registered",\n        "primary_question": (\n            "Does redistributing the already-adopted EXP047 top-1% negative loss mass "\n            "toward the top 0.1% improve natural genome-wide nucleotide AP?"\n        ),\n        "branch_point": str(BASE_CHECKPOINT),\n        "reference_model": "EXP047-B historical matched control",\n        "candidate": (\n            "same EXP045@50k continuation as EXP047-B, but old top-1% stratum is split "\n            "into top 0-0.1% and 0.1-1% while preserving total top-1% negative mass 0.13"\n        ),\n        "negative_strata": NEGATIVE_STRATA_NAMES,\n        "stratum_source": (\n            "EXP045 train-negative logits already saved in corrected EXP046B CV shards; "\n            "validation is excluded"\n        ),\n        "natural_negative_mixture": [0.001, 0.009, 0.04, 0.10, 0.85],\n        "candidate_negative_mixture": TARGET_NEGATIVE_MIXTURE.tolist(),\n        "candidate_negative_weight_multipliers_ideal": (\n            TARGET_NEGATIVE_MIXTURE\n            / np.asarray([0.001, 0.009, 0.04, 0.10, 0.85], dtype=np.float64)\n        ).tolist(),\n        "top1_total_mass_preserved": float(TARGET_NEGATIVE_MIXTURE[:2].sum()),\n        "continuation": {\n            "steps": CONTINUATION_STEPS,\n            "batch_size": BATCH_SIZE,\n            "warmup_steps": WARMUP_STEPS,\n            "start_lr": START_LR,\n            "peak_lr": PEAK_LR,\n            "end_lr": END_LR,\n            "schedule": "linear warm-up then cosine decay",\n        },\n        "gpu_policy": {\n            "type": "soft duty-cycle throttle after optimizer update",\n            "target": GPU_DUTY_CYCLE,\n            "hard_hardware_cap": False,\n        },\n        "primary_metric": "natural full-validation nucleotide AP after 5-decimal quantization",\n        "selection_rule": (\n            "ADOPT_ULTRA_TAIL only if candidate-EXP047B overall AP >= 0.010, "\n            "candidate improves AP on every validation contig, and overall Spearman "\n            "does not drop by more than 0.003; raw-AP and LOW_TP/top-budget audits remain mandatory"\n        ),\n        "validation_policy": "one full final candidate scan; no checkpoint selection on validation",\n        "checkpoint_policy": (\n            "atomic latest.pt every 500 steps; immutable snapshots at 2k/5k/10k/15k/20k"\n        ),\n        "scientific_caveat": (\n            "The validation split has been inspected in prior experiments; EXP048 is exploratory, "\n            "not a fresh unbiased estimate."\n        ),\n    }\n    if PREREG_PATH.exists():\n        existing = json.loads(PREREG_PATH.read_text(encoding="utf-8"))\n        expected = dict(payload)\n        expected["created"] = existing.get("created")\n        if existing != expected:\n            raise RuntimeError(f"Refusing to overwrite changed pre-registration: {PREREG_PATH}")\n    else:\n        atomic_json(PREREG_PATH, payload)\n    print(f"Pre-registration: {PREREG_PATH}")\n\n\ndef inspect_inputs() -> None:\n    required = [BASE_CHECKPOINT, BASE_METRICS, CV_SHARDS, EXP047_B_METRICS]\n    missing = [path for path in required if not path.exists()]\n    if missing:\n        raise FileNotFoundError(f"Missing required inputs: {missing}")\n    checkpoint = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)\n    required_keys = {\n        "model_state_dict", "optimizer_state_dict", "generator_rng_state", "torch_rng_state",\n        "model_config", "profile_config", "step",\n    }\n    absent = required_keys - set(checkpoint)\n    if absent:\n        raise ValueError(f"Base checkpoint misses keys: {sorted(absent)}")\n    assert int(checkpoint["step"]) == BASE_STEP\n    assert int(checkpoint["model_config"]["channels"]) == EXPECTED_CHANNELS\n    assert int(checkpoint["model_config"]["theoretical_rf"]) == EXPECTED_RF\n    model = create_cnn_presence_intensity(channels=EXPECTED_CHANNELS)\n    model.load_state_dict(checkpoint["model_state_dict"])\n    parameter_count = sum(parameter.numel() for parameter in model.parameters())\n    assert parameter_count == EXPECTED_PARAMETERS\n    fold_dirs = sorted(path for path in CV_SHARDS.glob("fold_*") if path.is_dir())\n    shard_files = [file for fold in fold_dirs for file in sorted((fold / "shards").glob("*.npz"))]\n    if len(fold_dirs) != 9 or not shard_files:\n        raise RuntimeError(f"Incomplete corrected CV shards: folds={len(fold_dirs)}, files={len(shard_files)}")\n    with np.load(shard_files[0], allow_pickle=False) as sample:\n        expected = {"region_key", "base_logit", "valid_mask", "positive_mask"}\n        if not expected <= set(sample.files):\n            raise ValueError(f"Shard misses arrays: {sorted(expected - set(sample.files))}")\n        assert sample["base_logit"].shape[1] == 32\n    print(f"Python: {sys.executable}")\n    print(f"Base checkpoint: step={checkpoint[\'step\']}, sha256={sha256_file(BASE_CHECKPOINT)}")\n    print(f"Model parameters: {parameter_count:,}")\n    print(f"Corrected train shards: folds={len(fold_dirs)}, files={len(shard_files):,}")\n    print(f"LR probes: step1={planned_lr(1):.8g}, warmup={planned_lr(WARMUP_STEPS):.8g}, final={planned_lr(CONTINUATION_STEPS):.8g}")\n\n\ndef iter_shards() -> list[Path]:\n    files: list[Path] = []\n    for fold in sorted(path for path in CV_SHARDS.glob("fold_*") if path.is_dir()):\n        files.extend(sorted((fold / "shards").glob("tiles_*.npz")))\n    return files\n\n\ndef prepare_risk_cache() -> None:\n    if RISK_CACHE.exists() and RISK_MANIFEST.exists():\n        manifest = json.loads(RISK_MANIFEST.read_text(encoding="utf-8"))\n        expected_bytes = int(np.prod(manifest["shape"], dtype=np.int64))\n        if RISK_CACHE.stat().st_size == expected_bytes and manifest["complete"]:\n            actual_sha = sha256_file(RISK_CACHE)\n            if "cache_sha256" not in manifest:\n                manifest["cache_sha256"] = actual_sha\n                atomic_json(RISK_MANIFEST, manifest)\n            elif manifest["cache_sha256"] != actual_sha:\n                raise RuntimeError("Completed risk cache checksum mismatch")\n            print(f"Complete risk cache already exists: {RISK_CACHE}")\n            return\n        raise RuntimeError("Risk cache/manifest exists but is incomplete or inconsistent")\n\n    RISK_CACHE_DIR.mkdir(parents=True, exist_ok=True)\n    split = load_split(ROOT)\n    sequences, _ = load_baseline_sequences(split)\n    base = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)\n    profile_config = ProfileWindowConfig(**base["profile_config"])\n    generator = ProfileWindowGenerator(\n        prepared_split=split, sequences=sequences, part="train", config=profile_config, seed=42\n    )\n    tiles = generator.tile_index\n    train_contigs = sorted(set(split.intervals("train")["contig"].astype(str)))\n    validation_contigs = sorted(set(split.intervals("validation")["contig"].astype(str)))\n    all_contigs = train_contigs + validation_contigs\n    code_to_contig = {index: contig for index, contig in enumerate(all_contigs)}\n    tile_lookup: dict[str, tuple[np.ndarray, np.ndarray]] = {}\n    for contig, frame in tiles.groupby(tiles["contig"].astype(str), sort=False):\n        group_ids = (frame["target_start"].to_numpy(np.int64) // 1024).astype(np.int64)\n        tile_ids = frame["tile_id"].to_numpy(np.int64)\n        order = np.argsort(group_ids, kind="stable")\n        tile_lookup[str(contig)] = (group_ids[order], tile_ids[order])\n\n    files = iter_shards()\n    histogram = np.zeros(SCORE_LEVELS, dtype=np.int64)\n    started = time.perf_counter()\n    for file_index, path in enumerate(files, start=1):\n        with np.load(path, allow_pickle=False) as data:\n            valid = unpack_mask(data["valid_mask"])\n            positive = unpack_mask(data["positive_mask"])\n            negative = valid & ~positive\n            bins = score_bins(data["base_logit"])\n            histogram += np.bincount(bins[negative], minlength=SCORE_LEVELS)\n        if file_index == 1 or file_index % 500 == 0 or file_index == len(files):\n            print(f"risk histogram: {file_index:,}/{len(files):,} shards", flush=True)\n    total_negative = int(histogram.sum())\n    if total_negative != generator.total_negative:\n        raise AssertionError(f"Negative population mismatch: shards={total_negative}, generator={generator.total_negative}")\n\n    cut_counts = np.asarray(\n        [round(total_negative * fraction) for fraction in NEGATIVE_RANK_CUT_FRACTIONS],\n        dtype=np.int64,\n    )\n    higher = np.cumsum(histogram[::-1], dtype=np.int64)[::-1] - histogram\n    seen_by_score = np.zeros(SCORE_LEVELS, dtype=np.int64)\n    shape = (len(tiles), 2, profile_config.target_length)\n    temp = RISK_CACHE.with_suffix(RISK_CACHE.suffix + ".tmp")\n    cache = np.memmap(temp, mode="w+", dtype=np.uint8, shape=shape)\n    cache[:] = np.uint8(255)\n    flat_cache = cache.reshape(-1)\n    realized = np.zeros(len(NEGATIVE_STRATA_NAMES), dtype=np.int64)\n    assigned = 0\n\n    for file_index, path in enumerate(files, start=1):\n        with np.load(path, allow_pickle=False) as data:\n            keys = data["region_key"].astype(np.uint64, copy=False)\n            valid = unpack_mask(data["valid_mask"])\n            positive = unpack_mask(data["positive_mask"])\n            negative = valid & ~positive\n            bins = score_bins(data["base_logit"])\n            strata = np.full(bins.shape, 255, dtype=np.uint8)\n            negative_scores = bins[negative]\n            order = np.argsort(negative_scores, kind="stable")\n            ordered_scores = negative_scores[order]\n            group_starts = np.r_[\n                np.int64(0),\n                np.flatnonzero(ordered_scores[1:] != ordered_scores[:-1]).astype(np.int64) + 1,\n            ]\n            group_ends = np.r_[group_starts[1:], np.int64(len(ordered_scores))]\n            group_lengths = group_ends - group_starts\n            unique_scores = ordered_scores[group_starts]\n            offsets = np.arange(len(ordered_scores), dtype=np.int64) - np.repeat(\n                group_starts, group_lengths\n            )\n            rank_bases = np.repeat(\n                higher[unique_scores] + seen_by_score[unique_scores], group_lengths\n            )\n            ordered_ranks = rank_bases + offsets\n            ordered_strata = np.searchsorted(\n                cut_counts, ordered_ranks, side="right"\n            ).astype(np.uint8)\n            stratum_values = np.empty(len(ordered_strata), dtype=np.uint8)\n            stratum_values[order] = ordered_strata\n            strata[negative] = stratum_values\n            seen_by_score[unique_scores] += group_lengths\n            realized += np.bincount(ordered_strata, minlength=len(NEGATIVE_STRATA_NAMES))\n\n            prefix = keys >> KEY_SHIFT\n            contig_codes = (prefix // np.uint64(2)).astype(np.int64)\n            strand_codes = (prefix % np.uint64(2)).astype(np.int64)\n            if len(np.unique(contig_codes)) != 1:\n                raise AssertionError(f"Mixed contigs in shard: {path}")\n            contig = code_to_contig[int(contig_codes[0])]\n            region_ids = (keys & REGION_MASK).astype(np.int64)\n            tile_groups = region_ids // 32\n            local_regions = region_ids % 32\n            known_groups, known_tile_ids = tile_lookup[contig]\n            positions = np.searchsorted(known_groups, tile_groups)\n            if np.any(positions >= len(known_groups)) or not np.array_equal(known_groups[positions], tile_groups):\n                raise AssertionError(f"Region-to-tile mapping failed: {path}")\n            tile_ids = known_tile_ids[positions]\n            destination = (\n                tile_ids[:, None] * (2 * profile_config.target_length)\n                + strand_codes[:, None] * profile_config.target_length\n                + local_regions[:, None] * 32\n                + BIT_SHIFTS[None, :].astype(np.int64)\n            )\n            selected_destination = destination[negative]\n            selected_strata = strata[negative]\n            if np.any(flat_cache[selected_destination] != 255):\n                raise AssertionError(f"Duplicate risk-cache destination: {path}")\n            flat_cache[selected_destination] = selected_strata\n            assigned += len(selected_strata)\n        if file_index == 1 or file_index % 500 == 0 or file_index == len(files):\n            print(f"risk assignment: {file_index:,}/{len(files):,} shards", flush=True)\n\n    cache.flush()\n    del flat_cache, cache\n    gc.collect()\n    if assigned != total_negative or int(realized.sum()) != total_negative:\n        raise AssertionError("Risk cache did not cover the complete train-negative population")\n    os.replace(temp, RISK_CACHE)\n    fractions = realized.astype(np.float64) / total_negative\n    if len(fractions) != len(TARGET_NEGATIVE_MIXTURE):\n        raise AssertionError((fractions, TARGET_NEGATIVE_MIXTURE))\n    multipliers = TARGET_NEGATIVE_MIXTURE / fractions\n    manifest = {\n        "complete": True,\n        "created": date.today().isoformat(),\n        "source_checkpoint": str(BASE_CHECKPOINT),\n        "source_checkpoint_sha256": sha256_file(BASE_CHECKPOINT),\n        "source_shard_root": str(CV_SHARDS),\n        "source_shard_count": len(files),\n        "score_definition": "sigmoid(EXP045 logit), rounded to 5 decimals",\n        "tie_break": "stable shard/path/row/position order within equal score",\n        "shape": list(shape),\n        "dtype": "uint8",\n        "cache_sha256": sha256_file(RISK_CACHE),\n        "invalid_or_positive_code": 255,\n        "rank_cut_counts": cut_counts.tolist(),\n        "stratum_counts": realized.tolist(),\n        "stratum_names": NEGATIVE_STRATA_NAMES,\n        "natural_fractions": fractions.tolist(),\n        "target_negative_mixture": TARGET_NEGATIVE_MIXTURE.tolist(),\n        "candidate_negative_weight_multipliers": multipliers.tolist(),\n        "total_train_negatives": total_negative,\n        "elapsed_seconds": time.perf_counter() - started,\n    }\n    atomic_json(RISK_MANIFEST, manifest)\n    print(json.dumps(manifest, indent=2))\n\n\ndef checkpoint_payload(\n    *, branch: str, local_step: int, model, optimizer, generator, history: list[dict],\n    elapsed_seconds: float, seen_position_strand: int, risk_manifest_sha256: str | None,\n) -> dict:\n    return {\n        "experiment": EXPERIMENT_ID,\n        "branch": branch,\n        "model_type": "presence_intensity",\n        "architecture": "rf1029_width512",\n        "base_experiment": BASE_EXPERIMENT,\n        "base_checkpoint": str(BASE_CHECKPOINT),\n        "base_checkpoint_sha256": sha256_file(BASE_CHECKPOINT),\n        "base_step": BASE_STEP,\n        "local_step": int(local_step),\n        "step": BASE_STEP + int(local_step),\n        "model_state_dict": model_state_on_cpu(model),\n        "optimizer_state_dict": tree_to_cpu(optimizer.state_dict()),\n        "generator_rng_state": copy.deepcopy(generator.rng.bit_generator.state),\n        "torch_rng_state": torch.get_rng_state(),\n        "model_config": {\n            "input_channels": 5, "channels": EXPECTED_CHANNELS,\n            "output_channels": 2, "theoretical_rf": EXPECTED_RF,\n        },\n        "profile_config": {\n            "target_length": generator.config.target_length,\n            "context_flank": generator.config.context_flank,\n            "positive_branch_fraction": generator.config.positive_branch_fraction,\n        },\n        "training_config": {\n            "continuation_steps": CONTINUATION_STEPS,\n            "batch_size": BATCH_SIZE,\n            "warmup_steps": WARMUP_STEPS,\n            "start_lr": START_LR,\n            "peak_lr": PEAK_LR,\n            "end_lr": END_LR,\n            "intensity_loss_weight": INTENSITY_LOSS_WEIGHT,\n            "presence_objective": (\n                "original_class_balanced_BCE" if branch == "A"\n                else "ultra_tail_five_strata_negative_BCE"\n            ),\n        },\n        "risk_manifest_sha256": risk_manifest_sha256,\n        "seen_position_strand": int(seen_position_strand),\n        "elapsed_seconds": float(elapsed_seconds),\n        "history": history,\n    }\n\n\ndef smoke(branch: str) -> None:\n    import torch_directml\n\n    if branch == "B" and not (RISK_CACHE.exists() and RISK_MANIFEST.exists()):\n        raise FileNotFoundError("Prepare risk cache before branch B smoke-test")\n    split = load_split(ROOT)\n    sequences, _ = load_baseline_sequences(split)\n    base = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)\n    config = ProfileWindowConfig(**base["profile_config"])\n    generator = ProfileWindowGenerator(\n        prepared_split=split, sequences=sequences, part="train", config=config, seed=42\n    )\n    generator.rng.bit_generator.state = copy.deepcopy(base["generator_rng_state"])\n    torch.set_rng_state(base["torch_rng_state"])\n    device = torch_directml.device()\n    model = create_cnn_presence_intensity(channels=EXPECTED_CHANNELS)\n    model.load_state_dict(base["model_state_dict"])\n    model = model.to(device)\n    optimizer = DirectMLAdam(model.parameters(), lr=planned_lr(1))\n    optimizer.load_state_dict(base["optimizer_state_dict"])\n    for group in optimizer.param_groups:\n        group["lr"] = planned_lr(1)\n    model.train()\n    batch = generator.sample_batch(BATCH_SIZE)\n    plus_weight = batch.plus_loss_weight\n    minus_weight = batch.minus_loss_weight_reverse\n    if branch == "B":\n        manifest = json.loads(RISK_MANIFEST.read_text(encoding="utf-8"))\n        cache = np.memmap(RISK_CACHE, mode="r", dtype=np.uint8, shape=tuple(manifest["shape"]))\n        strata = np.asarray(cache[batch.tile_id.numpy().astype(np.int64, copy=False)])\n        valid_negative = batch.mask.numpy() & ~batch.target.numpy().astype(bool)\n        assert not np.any(strata[valid_negative] == 255)\n        multiplier = np.ones(strata.shape, dtype=np.float32)\n        for stratum, value in enumerate(manifest["candidate_negative_weight_multipliers"]):\n            multiplier[strata == stratum] = np.float32(value)\n        plus_weight = plus_weight * torch.from_numpy(multiplier[:, 0:1, :])\n        minus_weight = minus_weight * torch.from_numpy(multiplier[:, 1:2, ::-1].copy())\n    plus_count = batch.count[:, 0:1, :].contiguous()\n    minus_count = batch.count[:, 1:2, :].flip(-1).contiguous()\n    optimizer.zero_grad(set_to_none=True)\n    with warnings.catch_warnings(record=True) as caught:\n        warnings.simplefilter("always")\n        plus_presence, minus_presence, plus_intensity, minus_intensity = (\n            forward_both_strands_multitask(\n                model, batch.x_both_strands.to(device), batch_size=BATCH_SIZE,\n                context_flank=config.context_flank, target_length=config.target_length,\n            )\n        )\n        presence_loss = dml_weighted_profile_bce_with_logits(\n            plus_presence, minus_presence, batch.plus_target.to(device),\n            batch.minus_target_reverse.to(device), plus_weight.to(device), minus_weight.to(device),\n        )\n        intensity_loss = (\n            dml_weighted_huber(\n                plus_intensity, torch.log1p(plus_count).to(device),\n                (2.0 * batch.plus_loss_weight * (plus_count > 0).float()).to(device), delta=1.0,\n            )\n            + dml_weighted_huber(\n                minus_intensity, torch.log1p(minus_count).to(device),\n                (2.0 * batch.minus_loss_weight_reverse * (minus_count > 0).float()).to(device), delta=1.0,\n            )\n        )\n        total_loss = presence_loss + INTENSITY_LOSS_WEIGHT * intensity_loss\n        values = [float(value.detach().cpu().item()) for value in (presence_loss, intensity_loss, total_loss)]\n        if not all(math.isfinite(value) for value in values):\n            raise FloatingPointError(values)\n        total_loss.backward()\n        optimizer.step()\n    fallbacks = [str(item.message) for item in caught if "fall back" in str(item.message).lower()]\n    if fallbacks:\n        raise RuntimeError("DirectML CPU fallback detected: " + " | ".join(fallbacks))\n    risk_hash = sha256_file(RISK_MANIFEST) if branch == "B" else None\n    preflight_checkpoint = RUN_DIR / f"_preflight_branch_{branch}.pt"\n    payload = checkpoint_payload(\n        branch=branch, local_step=1, model=model, optimizer=optimizer, generator=generator,\n        history=[], elapsed_seconds=0.0, seen_position_strand=int(batch.mask.sum()),\n        risk_manifest_sha256=risk_hash,\n    )\n    atomic_torch_save(preflight_checkpoint, payload)\n    restored = torch.load(preflight_checkpoint, map_location="cpu", weights_only=False)\n    assert restored["experiment"] == EXPERIMENT_ID\n    assert restored["branch"] == branch\n    assert restored["local_step"] == 1\n    preflight_checkpoint.unlink()\n    print(\n        f"SMOKE OK branch={branch} device={torch_directml.device_name(0)} "\n        f"presence={values[0]:.6f} intensity={values[1]:.6f} total={values[2]:.6f} "\n        "atomic_checkpoint=OK"\n    )\n\n\ndef train(branch: str) -> None:\n    import torch_directml\n\n    if branch == "B" and not (RISK_CACHE.exists() and RISK_MANIFEST.exists()):\n        raise FileNotFoundError("Prepare risk cache before training branch B")\n    current_base_sha = sha256_file(BASE_CHECKPOINT)\n    current_risk_hash = sha256_file(RISK_MANIFEST) if branch == "B" else None\n    output = branch_dir(branch)\n    output.mkdir(parents=True, exist_ok=True)\n    final_path = output / f"branch_{branch}_final.pt"\n    if final_path.exists():\n        checkpoint = torch.load(final_path, map_location="cpu", weights_only=False)\n        if int(checkpoint["local_step"]) == CONTINUATION_STEPS:\n            print(f"Branch {branch} already complete: {final_path}")\n            return\n    acquire_process_lock(output / "training.lock", f"train_branch_{branch}")\n\n    split = load_split(ROOT)\n    sequences, _ = load_baseline_sequences(split)\n    base = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)\n    profile_config = ProfileWindowConfig(**base["profile_config"])\n    generator = ProfileWindowGenerator(\n        prepared_split=split, sequences=sequences, part="train", config=profile_config, seed=42\n    )\n    device = torch_directml.device()\n    model = create_cnn_presence_intensity(channels=EXPECTED_CHANNELS)\n    latest = output / "latest.pt"\n    if latest.exists():\n        resume = torch.load(latest, map_location="cpu", weights_only=False)\n        if resume["branch"] != branch or resume["experiment"] != EXPERIMENT_ID:\n            raise RuntimeError(f"Wrong resume checkpoint: {latest}")\n        if resume["base_checkpoint_sha256"] != current_base_sha:\n            raise RuntimeError("Base checkpoint changed since branch training started")\n        if resume.get("risk_manifest_sha256") != current_risk_hash:\n            raise RuntimeError("Risk manifest changed since branch training started")\n        model.load_state_dict(resume["model_state_dict"])\n        generator.rng.bit_generator.state = copy.deepcopy(resume["generator_rng_state"])\n        torch.set_rng_state(resume["torch_rng_state"])\n        start_step = int(resume["local_step"])\n        history = list(resume["history"])\n        seen = int(resume["seen_position_strand"])\n        previous_elapsed = float(resume["elapsed_seconds"])\n        print(f"Resume branch {branch} at local step {start_step}")\n    else:\n        model.load_state_dict(base["model_state_dict"])\n        generator.rng.bit_generator.state = copy.deepcopy(base["generator_rng_state"])\n        torch.set_rng_state(base["torch_rng_state"])\n        start_step = 0\n        history = []\n        seen = 0\n        previous_elapsed = 0.0\n    model = model.to(device)\n    optimizer = DirectMLAdam(model.parameters(), lr=START_LR)\n    if latest.exists():\n        optimizer.load_state_dict(resume["optimizer_state_dict"])\n        del resume\n    else:\n        optimizer.load_state_dict(base["optimizer_state_dict"])\n    del base\n    model.train()\n\n    risk_cache = None\n    risk_multipliers = None\n    risk_hash = None\n    if branch == "B":\n        manifest = json.loads(RISK_MANIFEST.read_text(encoding="utf-8"))\n        if manifest["source_checkpoint_sha256"] != current_base_sha:\n            raise RuntimeError("Risk cache was built from a different base checkpoint")\n        if tuple(manifest["shape"]) != (\n            len(generator.tile_index), 2, profile_config.target_length\n        ):\n            raise RuntimeError("Risk cache shape does not match the training generator")\n        if RISK_CACHE.stat().st_size != int(np.prod(manifest["shape"], dtype=np.int64)):\n            raise RuntimeError("Risk cache file size is inconsistent with its manifest")\n        if manifest.get("cache_sha256") != sha256_file(RISK_CACHE):\n            raise RuntimeError("Risk cache checksum does not match its manifest")\n        risk_hash = current_risk_hash\n        risk_cache = np.memmap(\n            RISK_CACHE, mode="r", dtype=np.uint8, shape=tuple(manifest["shape"])\n        )\n        risk_multipliers = np.asarray(\n            manifest["candidate_negative_weight_multipliers"], dtype=np.float32\n        )\n\n    started = time.perf_counter()\n    ema_presence = history[-1]["ema_presence"] if history else None\n    ema_intensity = history[-1]["ema_intensity"] if history else None\n    ema_total = history[-1]["ema_total"] if history else None\n    fallback_messages: list[str] = []\n\n    for local_step in range(start_step + 1, CONTINUATION_STEPS + 1):\n        lr = planned_lr(local_step)\n        for group in optimizer.param_groups:\n            group["lr"] = lr\n        batch = generator.sample_batch(BATCH_SIZE)\n        plus_weight = batch.plus_loss_weight\n        minus_weight_reverse = batch.minus_loss_weight_reverse\n        if branch == "B":\n            tile_ids = batch.tile_id.numpy().astype(np.int64, copy=False)\n            strata = np.asarray(risk_cache[tile_ids])\n            valid_negative = batch.mask.numpy() & ~(batch.target.numpy().astype(bool))\n            if np.any(strata[valid_negative] == 255):\n                raise AssertionError("Risk cache misses a sampled valid negative")\n            multiplier = np.ones(strata.shape, dtype=np.float32)\n            for stratum, value in enumerate(risk_multipliers):\n                multiplier[strata == stratum] = value\n            plus_weight = plus_weight * torch.from_numpy(multiplier[:, 0:1, :])\n            minus_weight_reverse = minus_weight_reverse * torch.from_numpy(\n                multiplier[:, 1:2, ::-1].copy()\n            )\n\n        gpu_phase_started = time.perf_counter()\n        x_gpu = batch.x_both_strands.to(device)\n        plus_target_gpu = batch.plus_target.to(device)\n        minus_target_gpu = batch.minus_target_reverse.to(device)\n        plus_weight_gpu = plus_weight.to(device)\n        minus_weight_gpu = minus_weight_reverse.to(device)\n        plus_count = batch.count[:, 0:1, :].contiguous()\n        minus_count_reverse = batch.count[:, 1:2, :].flip(-1).contiguous()\n        plus_intensity_target = torch.log1p(plus_count).to(device)\n        minus_intensity_target = torch.log1p(minus_count_reverse).to(device)\n        plus_intensity_weight = (\n            2.0 * batch.plus_loss_weight * (plus_count > 0).float()\n        ).to(device)\n        minus_intensity_weight = (\n            2.0 * batch.minus_loss_weight_reverse * (minus_count_reverse > 0).float()\n        ).to(device)\n\n        optimizer.zero_grad(set_to_none=True)\n        with warnings.catch_warnings(record=True) as caught:\n            warnings.simplefilter("always")\n            plus_presence, minus_presence, plus_intensity, minus_intensity = (\n                forward_both_strands_multitask(\n                    model, x_gpu, batch_size=BATCH_SIZE,\n                    context_flank=profile_config.context_flank,\n                    target_length=profile_config.target_length,\n                )\n            )\n            presence_loss = dml_weighted_profile_bce_with_logits(\n                plus_presence, minus_presence, plus_target_gpu, minus_target_gpu,\n                plus_weight_gpu, minus_weight_gpu,\n            )\n            intensity_loss = (\n                dml_weighted_huber(\n                    plus_intensity, plus_intensity_target, plus_intensity_weight, delta=1.0\n                )\n                + dml_weighted_huber(\n                    minus_intensity, minus_intensity_target, minus_intensity_weight, delta=1.0\n                )\n            )\n            total_loss = presence_loss + INTENSITY_LOSS_WEIGHT * intensity_loss\n            values = [float(value.detach().cpu().item()) for value in (presence_loss, intensity_loss, total_loss)]\n            if not all(math.isfinite(value) for value in values):\n                raise FloatingPointError(f"Non-finite loss at branch {branch} step {local_step}: {values}")\n            total_loss.backward()\n            optimizer.step()\n            soft_gpu_duty_pause(time.perf_counter() - gpu_phase_started)\n        new_fallback = [\n            str(item.message) for item in caught if "fall back" in str(item.message).lower()\n        ]\n        if new_fallback:\n            fallback_messages.extend(new_fallback)\n            raise RuntimeError("DirectML CPU fallback detected: " + " | ".join(new_fallback))\n\n        presence_value, intensity_value, total_value = values\n        if ema_presence is None:\n            ema_presence, ema_intensity, ema_total = values\n        else:\n            ema_presence = 0.99 * ema_presence + 0.01 * presence_value\n            ema_intensity = 0.99 * ema_intensity + 0.01 * intensity_value\n            ema_total = 0.99 * ema_total + 0.01 * total_value\n        seen += int(batch.mask.sum())\n        elapsed = previous_elapsed + time.perf_counter() - started\n        if local_step % LOG_EVERY == 0 or local_step == CONTINUATION_STEPS:\n            run_elapsed = max(elapsed - previous_elapsed, 1e-9)\n            completed_now = local_step - start_step\n            sec_per_step = run_elapsed / completed_now\n            record = {\n                "branch": branch,\n                "local_step": local_step,\n                "step": BASE_STEP + local_step,\n                "learning_rate": lr,\n                "presence_loss": presence_value,\n                "intensity_loss": intensity_value,\n                "total_loss": total_value,\n                "ema_presence": float(ema_presence),\n                "ema_intensity": float(ema_intensity),\n                "ema_total": float(ema_total),\n                "elapsed_seconds": elapsed,\n                "seconds_per_step_current_process": sec_per_step,\n                "estimated_hours_remaining": (CONTINUATION_STEPS - local_step) * sec_per_step / 3600,\n                "seen_position_strand": seen,\n            }\n            history.append(record)\n            atomic_csv(output / "training_history.csv", pd.DataFrame(history))\n            print(\n                f"[{branch}] local={local_step:05d}/{CONTINUATION_STEPS} global={BASE_STEP + local_step} "\n                f"lr={lr:.7g} loss=({presence_value:.5f},{intensity_value:.5f},{total_value:.5f}) "\n                f"sec/step={sec_per_step:.3f} ETA={record[\'estimated_hours_remaining\']:.2f}h",\n                flush=True,\n            )\n        if local_step % LATEST_EVERY == 0 or local_step == CONTINUATION_STEPS:\n            payload = checkpoint_payload(\n                branch=branch, local_step=local_step, model=model, optimizer=optimizer,\n                generator=generator, history=history, elapsed_seconds=elapsed,\n                seen_position_strand=seen, risk_manifest_sha256=risk_hash,\n            )\n            atomic_torch_save(latest, payload)\n            if local_step in SNAPSHOT_STEPS:\n                atomic_torch_save(output / f"model_local_step_{local_step:05d}.pt", payload)\n            if local_step == CONTINUATION_STEPS:\n                atomic_torch_save(final_path, payload)\n            del payload\n            gc.collect()\n\n        del (\n            batch, x_gpu, plus_target_gpu, minus_target_gpu, plus_weight_gpu, minus_weight_gpu,\n            plus_count, minus_count_reverse, plus_intensity_target, minus_intensity_target,\n            plus_intensity_weight, minus_intensity_weight, plus_presence, minus_presence,\n            plus_intensity, minus_intensity, presence_loss, intensity_loss, total_loss,\n        )\n\n    summary = {\n        "branch": branch,\n        "complete": True,\n        "local_steps": CONTINUATION_STEPS,\n        "global_step": BASE_STEP + CONTINUATION_STEPS,\n        "training_hours": elapsed / 3600,\n        "final_lr": planned_lr(CONTINUATION_STEPS),\n        "fallback_warnings": fallback_messages,\n        "final_checkpoint": str(final_path),\n    }\n    atomic_json(output / "training_complete.json", summary)\n    print(json.dumps(summary, indent=2))\n\n\ndef score_fixed_cohorts(model, generator, device, branch: str) -> pd.DataFrame:\n    paths = {\n        "HARD_FP": ORACLE_RUN / "fixed_hard_fp_coordinates.csv",\n        "LOW_TP": ORACLE_RUN / "fixed_low_tp_coordinates.csv",\n    }\n    frames = []\n    for cohort, path in paths.items():\n        if path.is_file():\n            frame = pd.read_csv(path)\n            frame = frame[["contig", "strand", "position", "baseline_score"]].copy()\n            frame["cohort"] = cohort\n            frames.append(frame)\n    if not frames:\n        return pd.DataFrame()\n    cohorts = pd.concat(frames, ignore_index=True)\n    cohorts["target_start"] = cohorts["position"] // 1024 * 1024\n    tile_map = {\n        (str(row.contig), int(row.target_start)): index\n        for index, row in generator.tile_index.iterrows()\n    }\n    cohorts["tile_index"] = [\n        tile_map[(str(contig), int(start))]\n        for contig, start in zip(cohorts["contig"], cohorts["target_start"])\n    ]\n    scores = np.full(len(cohorts), np.nan, dtype=np.float32)\n    groups = cohorts.groupby("tile_index", sort=True).groups\n    tile_groups = list(groups.items())\n    cohort_batch_size = 4\n    model.eval()\n    with torch.no_grad():\n        for start in range(0, len(tile_groups), cohort_batch_size):\n            chunk = tile_groups[start : start + cohort_batch_size]\n            tile_indices = [int(tile_index) for tile_index, _ in chunk]\n            selected = generator.tile_index.iloc[tile_indices]\n            batch = generator.make_batch(selected)\n            x = batch.x_both_strands.to(device)\n            plus, minus_reverse, _, _ = forward_both_strands_multitask(\n                model, x, batch_size=len(chunk), context_flank=generator.config.context_flank,\n                target_length=generator.config.target_length,\n            )\n            plus_probability = sigmoid_numpy(plus[:, 0].detach().cpu().numpy())\n            minus_probability = sigmoid_numpy(minus_reverse[:, 0].detach().cpu().numpy()[:, ::-1])\n            for local_batch, (_, row_ids) in enumerate(chunk):\n                for row_id in row_ids:\n                    row = cohorts.loc[row_id]\n                    local = int(row["position"] - row["target_start"])\n                    scores[row_id] = (\n                        plus_probability[local_batch, local]\n                        if row["strand"] == "+"\n                        else minus_probability[local_batch, local]\n                    )\n            completed = min(start + len(chunk), len(tile_groups))\n            if start == 0 or completed % 1000 < len(chunk) or completed == len(tile_groups):\n                print(f"[{branch}] cohort tiles {completed:,}/{len(tile_groups):,}", flush=True)\n            del batch, x, plus, minus_reverse\n    if not np.all(np.isfinite(scores)):\n        raise AssertionError("Non-finite cohort scores")\n    cohorts[f"score_{branch}"] = scores\n    return cohorts.drop(columns=["tile_index", "target_start"])\n\n\ndef evaluate(branch: str) -> None:\n    import torch_directml\n\n    output = branch_dir(branch)\n    final_path = output / f"branch_{branch}_final.pt"\n    if not final_path.is_file():\n        raise FileNotFoundError(f"Final checkpoint missing: {final_path}")\n    metrics_path = output / "validation_metrics_report.csv"\n    if metrics_path.exists() and (output / "evaluation_complete.json").exists():\n        print(f"Branch {branch} evaluation already complete: {metrics_path}")\n        return\n    acquire_process_lock(output / "evaluation.lock", f"evaluate_branch_{branch}")\n    checkpoint = torch.load(final_path, map_location="cpu", weights_only=False)\n    device = torch_directml.device()\n    model = create_cnn_presence_intensity(channels=EXPECTED_CHANNELS)\n    model.load_state_dict(checkpoint["model_state_dict"])\n    model = model.to(device)\n    model.eval()\n    split = load_split(ROOT)\n    sequences, _ = load_baseline_sequences(split)\n    config = ProfileWindowConfig(**checkpoint["profile_config"])\n    generator = ProfileWindowGenerator(\n        prepared_split=split, sequences=sequences, part="validation", config=config, seed=42\n    )\n    result = evaluate_presence_intensity_model(\n        model, generator, device=device, batch_size=1, score_decimals=5,\n        progress_every_batches=100, fail_on_cpu_fallback=True,\n    )\n    atomic_csv(output / "validation_all_scores.csv", result.metrics)\n    presence = result.metrics.loc[result.metrics["score"] == "presence_probability"].copy()\n    primary = presence[["segment", "average_precision", "epic_spearman"]].copy()\n    atomic_csv(metrics_path, primary)\n    intensity = result.metrics.loc[result.metrics["score"] == "intensity_prediction", ["segment", "epic_spearman"]]\n    intensity = intensity.rename(columns={"epic_spearman": "intensity_EPIC_Spearman_raw"})\n    strong = presence[["segment", "positives", "average_precision", "epic_spearman", "quantized_spearman"]].rename(\n        columns={\n            "positives": "positive_n", "average_precision": "presence_AP_5dp",\n            "epic_spearman": "presence_EPIC_Spearman_raw",\n            "quantized_spearman": "presence_EPIC_Spearman_5dp",\n        }\n    ).merge(intensity, on="segment", validate="one_to_one")\n    atomic_csv(output / "strong_validation_metrics.csv", strong)\n    cohorts = score_fixed_cohorts(model, generator, device, branch)\n    if not cohorts.empty:\n        atomic_csv(output / "fixed_cohort_scores.csv", cohorts)\n        score_column = f"score_{branch}"\n        rows = []\n        for cohort, frame in cohorts.groupby("cohort", sort=False):\n            values = frame[score_column].to_numpy(np.float64)\n            rows.append({\n                "branch": branch, "cohort": cohort, "n": len(values),\n                "mean": values.mean(), "p10": np.quantile(values, 0.1),\n                "p50": np.quantile(values, 0.5), "p90": np.quantile(values, 0.9),\n                "p99": np.quantile(values, 0.99),\n            })\n        atomic_csv(output / "fixed_cohort_score_summary.csv", pd.DataFrame(rows))\n    evaluation = {\n        "branch": branch, "complete": True,\n        "elapsed_seconds": result.elapsed_seconds,\n        "positions_per_second": result.positions_per_second,\n        "processed_tiles": result.processed_tiles,\n        "processed_positions": result.processed_positions,\n        "processed_positives": result.processed_positives,\n        "score_decimals": result.score_decimals,\n        "fallback_warnings": result.fallback_warnings,\n    }\n    atomic_json(output / "evaluation_complete.json", evaluation)\n    print(primary.to_string(index=False))\n\n\ndef _font(size: int, *, bold: bool = False):\n    candidates = (\n        "C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf",\n        "C:/Windows/Fonts/segoeuib.ttf" if bold else "C:/Windows/Fonts/segoeui.ttf",\n    )\n    for candidate in candidates:\n        if Path(candidate).is_file():\n            return ImageFont.truetype(candidate, size=size)\n    return ImageFont.load_default()\n\n\ndef _draw_vertical_label(image: Image.Image, text: str, font) -> None:\n    probe = ImageDraw.Draw(image)\n    box = probe.textbbox((0, 0), text, font=font)\n    label = Image.new("RGBA", (box[2] - box[0] + 8, box[3] - box[1] + 8), (255, 255, 255, 0))\n    ImageDraw.Draw(label).text((4, 4), text, fill="#202124", font=font, anchor="la")\n    rotated = label.rotate(90, expand=True)\n    image.paste(rotated, (12, (image.height - rotated.height) // 2), rotated)\n\n\ndef _save_line_plot(path: Path, series: list[tuple[str, np.ndarray, np.ndarray]], title: str, y_label: str) -> None:\n    width, height = 1200, 620\n    left, top, right, bottom = 105, 75, 35, 85\n    image = Image.new("RGB", (width, height), "white")\n    draw = ImageDraw.Draw(image)\n    title_font, label_font, small_font = _font(27, bold=True), _font(18), _font(15)\n    colors = ["#1967D2", "#D93025", "#188038", "#9334E6"]\n    x_values = np.concatenate([np.asarray(item[1], dtype=np.float64) for item in series])\n    y_values = np.concatenate([np.asarray(item[2], dtype=np.float64) for item in series])\n    x_min, x_max = float(x_values.min()), float(x_values.max())\n    y_min, y_max = float(y_values.min()), float(y_values.max())\n    if math.isclose(x_min, x_max):\n        x_max = x_min + 1.0\n    if math.isclose(y_min, y_max):\n        y_max = y_min + 1.0\n    padding = 0.08 * (y_max - y_min)\n    y_min, y_max = y_min - padding, y_max + padding\n    plot_width, plot_height = width - left - right, height - top - bottom\n\n    def transform(x, y):\n        px = left + (x - x_min) / (x_max - x_min) * plot_width\n        py = top + (y_max - y) / (y_max - y_min) * plot_height\n        return float(px), float(py)\n\n    draw.text((width / 2, 25), title, fill="#202124", font=title_font, anchor="ma")\n    for tick in range(6):\n        fraction = tick / 5\n        py = top + fraction * plot_height\n        value = y_max - fraction * (y_max - y_min)\n        draw.line((left, py, width - right, py), fill="#E8EAED", width=1)\n        draw.text((left - 12, py), f"{value:.4f}", fill="#5F6368", font=small_font, anchor="rm")\n    draw.line((left, top, left, height - bottom), fill="#3C4043", width=2)\n    draw.line((left, height - bottom, width - right, height - bottom), fill="#3C4043", width=2)\n    for tick in range(5):\n        fraction = tick / 4\n        px = left + fraction * plot_width\n        value = x_min + fraction * (x_max - x_min)\n        draw.text((px, height - bottom + 12), f"{value:,.0f}", fill="#5F6368", font=small_font, anchor="ma")\n    draw.text((width / 2, height - 28), "Continuation step", fill="#202124", font=label_font, anchor="ma")\n    _draw_vertical_label(image, y_label, label_font)\n    for index, (label, xs, ys) in enumerate(series):\n        points = [transform(float(x), float(y)) for x, y in zip(xs, ys)]\n        if len(points) > 1:\n            draw.line(points, fill=colors[index % len(colors)], width=4, joint="curve")\n        for point in points[:: max(1, len(points) // 20)]:\n            x, y = point\n            draw.ellipse((x - 3, y - 3, x + 3, y + 3), fill=colors[index % len(colors)])\n        legend_x = left + index * 210\n        draw.line((legend_x, 58, legend_x + 35, 58), fill=colors[index % len(colors)], width=4)\n        draw.text((legend_x + 43, 58), label, fill="#202124", font=small_font, anchor="lm")\n    path.parent.mkdir(parents=True, exist_ok=True)\n    image.save(path)\n\n\ndef _save_grouped_bars(\n    path: Path, categories: list[str], series: list[tuple[str, list[float]]], title: str, y_label: str\n) -> None:\n    width, height = 1200, 650\n    left, top, right, bottom = 105, 80, 35, 125\n    image = Image.new("RGB", (width, height), "white")\n    draw = ImageDraw.Draw(image)\n    title_font, label_font, small_font = _font(27, bold=True), _font(18), _font(15)\n    colors = ["#9AA0A6", "#1967D2", "#D93025", "#188038"]\n    maximum = max(float(value) for _, values in series for value in values)\n    y_max = max(maximum * 1.15, 1e-6)\n    plot_width, plot_height = width - left - right, height - top - bottom\n    draw.text((width / 2, 28), title, fill="#202124", font=title_font, anchor="ma")\n    for tick in range(6):\n        fraction = tick / 5\n        py = top + fraction * plot_height\n        value = y_max * (1.0 - fraction)\n        draw.line((left, py, width - right, py), fill="#E8EAED", width=1)\n        draw.text((left - 12, py), f"{value:.3f}", fill="#5F6368", font=small_font, anchor="rm")\n    draw.line((left, top, left, height - bottom), fill="#3C4043", width=2)\n    draw.line((left, height - bottom, width - right, height - bottom), fill="#3C4043", width=2)\n    group_width = plot_width / max(len(categories), 1)\n    bar_area = group_width * 0.72\n    bar_width = bar_area / max(len(series), 1)\n    for category_index, category in enumerate(categories):\n        center = left + (category_index + 0.5) * group_width\n        draw.text((center, height - bottom + 18), category, fill="#202124", font=small_font, anchor="ma")\n        for series_index, (_, values) in enumerate(series):\n            value = float(values[category_index])\n            x0 = center - bar_area / 2 + series_index * bar_width + 2\n            x1 = x0 + bar_width - 4\n            y0 = top + (y_max - value) / y_max * plot_height\n            draw.rectangle((x0, y0, x1, height - bottom), fill=colors[series_index % len(colors)])\n            draw.text(((x0 + x1) / 2, y0 - 5), f"{value:.4f}", fill="#202124", font=_font(12), anchor="ms")\n    legend_x = left\n    for index, (label, _) in enumerate(series):\n        x = legend_x + index * 215\n        draw.rectangle((x, 55, x + 24, 70), fill=colors[index % len(colors)])\n        draw.text((x + 32, 63), label, fill="#202124", font=small_font, anchor="lm")\n    _draw_vertical_label(image, y_label, label_font)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    image.save(path)\n\n\ndef _write_report_and_registry(\n    *, decision: str, comparison: pd.DataFrame, overall: pd.Series, metrics_a: pd.DataFrame,\n    metrics_b: pd.DataFrame, plot_dir: Path,\n) -> tuple[Path, Path]:\n    adopt = decision == "ADOPT_RISK_FOCUSED_LOSS"\n    card_path = ROOT / "cnn_exp" / f"{EXPERIMENT_ID}.md"\n    rows = []\n    for row in comparison.itertuples(index=False):\n        rows.append(\n            f"| {row.segment} | {row.ap_EXP045:.9f} | {row.ap_A:.9f} | "\n            f"{row.ap_B:.9f} | {row.delta_B_vs_A:+.9f} |"\n        )\n    report = f"""## Итоговое сравнение\n\n| Segment | EXP045 AP | A control AP | B risk-focused AP | B−A |\n|---|---:|---:|---:|---:|\n{chr(10).join(rows)}\n\n- Pre-registered decision: **{decision}**.\n- B−A overall AP: `{float(overall[\'delta_B_vs_A\']):+.9f}`.\n- B−EXP045 overall AP: `{float(overall[\'delta_B_vs_EXP045\']):+.9f}`.\n- AP 0.20 reached: `{float(overall[\'ap_B\']) >= 0.20}`.\n- Risk-loss selection rule satisfied: `{adopt}`.\n\n### Графики\n\n![[{plot_dir.relative_to(ROOT).as_posix()}/01_training_losses.png]]\n\n![[{plot_dir.relative_to(ROOT).as_posix()}/02_overall_comparison.png]]\n\n![[{plot_dir.relative_to(ROOT).as_posix()}/03_per_contig_comparison.png]]\n\n![[{plot_dir.relative_to(ROOT).as_posix()}/04_paired_training_losses.png]]\n\n![[{plot_dir.relative_to(ROOT).as_posix()}/05_exp045_vs_A_vs_B_ap.png]]\n\n### Вывод\n\nВетка B сравнивается с matched continuation A, поэтому дополнительное обучение отделено от эффекта risk-focused loss. Validation остаётся exploratory: этот split просматривался в предыдущих экспериментах.\n"""\n    text = card_path.read_text(encoding="utf-8")\n    start_marker = "<!-- auto:cnn-experiment-report:start -->"\n    end_marker = "<!-- auto:cnn-experiment-report:end -->"\n    if start_marker not in text or end_marker not in text:\n        raise ValueError(f"Auto-report markers missing in {card_path}")\n    before, remainder = text.split(start_marker, 1)\n    _, after = remainder.split(end_marker, 1)\n    text = before + start_marker + "\\n\\n" + report + "\\n" + end_marker + after\n    text = re.sub(r"(?m)^status: .+$", "status: completed", text, count=1)\n    text = re.sub(r"(?m)^decision: .+$", f"decision: {\'adopt\' if adopt else \'reject\'}", text, count=1)\n    card_path.write_text(text, encoding="utf-8", newline="\\n")\n\n    summary_path = RUN_DIR / "experiment_summary.json"\n    atomic_json(summary_path, {\n        "experiment_id": EXPERIMENT_ID,\n        "title": "paired WIDTH512 risk-focused loss continuation",\n        "hypothesis": "Risk-focused end-to-end presence training improves natural nucleotide AP over a matched continuation",\n        "reference_experiment": "CNN-EXP-047-A",\n        "decision": "adopt" if adopt else "reject",\n        "date": date.today().isoformat(),\n        "notebook": NOTEBOOK_REL,\n        "reference_ap": float(overall["ap_A"]),\n        "candidate_ap": float(overall["ap_B"]),\n        "delta_ap": float(overall["delta_B_vs_A"]),\n        "relative_ap": float(overall["ap_B"] / overall["ap_A"] - 1.0),\n        "reference_spearman": float(overall["spearman_A"]),\n        "candidate_spearman": float(overall["spearman_B"]),\n        "delta_spearman": float(overall["spearman_B"] - overall["spearman_A"]),\n    })\n\n    index_path = ROOT / "cnn_exp" / "_index.md"\n    lines = index_path.read_text(encoding="utf-8").splitlines()\n    row = (\n        f"| [[cnn_exp/{EXPERIMENT_ID}.md\\\\|{EXPERIMENT_ID}]] | paired WIDTH512 risk-focused loss continuation | "\n        f"{float(overall[\'ap_B\']):.9f} | {float(overall[\'spearman_B\']):.9f} | "\n        f"`{\'adopt\' if adopt else \'reject\'}` |"\n    )\n    prefix = f"| [[cnn_exp/{EXPERIMENT_ID}.md"\n    for index, line in enumerate(lines):\n        if line.startswith(prefix):\n            lines[index] = row\n            break\n    else:\n        table_rows = [index for index, line in enumerate(lines) if line.startswith("| [[cnn_exp/CNN-EXP-")]\n        if not table_rows:\n            raise ValueError("CNN registry table was not found")\n        lines.insert(table_rows[-1] + 1, row)\n    index_path.write_text("\\n".join(lines) + "\\n", encoding="utf-8", newline="\\n")\n    return card_path, summary_path\n\n\ndef finalize() -> None:\n    preregister()\n    candidate_path = branch_dir("B") / "validation_metrics_report.csv"\n    if not candidate_path.is_file():\n        raise FileNotFoundError(f"Evaluate candidate B first: {candidate_path}")\n    if not EXP047_B_METRICS.is_file():\n        raise FileNotFoundError(f"Historical EXP047-B metrics missing: {EXP047_B_METRICS}")\n\n    candidate = pd.read_csv(candidate_path).rename(\n        columns={"average_precision": "ap_EXP048", "epic_spearman": "spearman_EXP048"}\n    )\n    reference = pd.read_csv(EXP047_B_METRICS).rename(\n        columns={"average_precision": "ap_EXP047B", "epic_spearman": "spearman_EXP047B"}\n    )\n    comparison = reference.merge(candidate, on="segment", validate="one_to_one")\n    comparison["delta_ap"] = comparison["ap_EXP048"] - comparison["ap_EXP047B"]\n    comparison["delta_spearman"] = comparison["spearman_EXP048"] - comparison["spearman_EXP047B"]\n    atomic_csv(RUN_DIR / "EXP048_vs_EXP047B.csv", comparison)\n\n    strong_src = branch_dir("B") / "strong_validation_metrics.csv"\n    if strong_src.is_file():\n        shutil.copy2(strong_src, RUN_DIR / "strong_validation_metrics.csv")\n    hist_src = branch_dir("B") / "training_history.csv"\n    if hist_src.is_file():\n        shutil.copy2(hist_src, RUN_DIR / "training_history.csv")\n\n    overall = comparison.loc[comparison["segment"] == "overall"].iloc[0]\n    per_contig = comparison.loc[comparison["segment"] != "overall"]\n    quantitative_gate = (\n        float(overall["delta_ap"]) >= 0.010\n        and bool((per_contig["delta_ap"] > 0).all())\n        and float(overall["delta_spearman"]) >= -0.003\n    )\n\n    cohort_comparison_path = None\n    candidate_cohort = branch_dir("B") / "fixed_cohort_scores.csv"\n    if EXP047_B_COHORT.is_file() and candidate_cohort.is_file():\n        left = pd.read_csv(EXP047_B_COHORT).rename(columns={"score_B": "score_EXP047B"})\n        right = pd.read_csv(candidate_cohort).rename(columns={"score_B": "score_EXP048"})\n        keys = ["contig", "strand", "position", "baseline_score", "cohort"]\n        cohort = left[keys + ["score_EXP047B"]].merge(\n            right[keys + ["score_EXP048"]], on=keys, validate="one_to_one"\n        )\n        cohort["delta_EXP048_vs_EXP047B"] = cohort["score_EXP048"] - cohort["score_EXP047B"]\n        cohort_comparison_path = RUN_DIR / "fixed_cohort_EXP047B_vs_EXP048.csv"\n        atomic_csv(cohort_comparison_path, cohort)\n        summary = cohort.groupby("cohort", as_index=False).agg(\n            n=("delta_EXP048_vs_EXP047B", "size"),\n            mean_score_EXP047B=("score_EXP047B", "mean"),\n            mean_score_EXP048=("score_EXP048", "mean"),\n            mean_delta=("delta_EXP048_vs_EXP047B", "mean"),\n        )\n        atomic_csv(RUN_DIR / "fixed_cohort_EXP047B_vs_EXP048_summary.csv", summary)\n\n    result = {\n        "experiment_id": EXPERIMENT_ID,\n        "operational_reference": "EXP047-B",\n        "quantitative_adopt_gate_passed": bool(quantitative_gate),\n        "final_adopt_pending_raw_ap_and_budget_recall_audit": True,\n        "overall": {key: float(overall[key]) for key in comparison.columns if key != "segment"},\n        "per_contig": comparison.loc[comparison["segment"] != "overall"].to_dict(orient="records"),\n        "gpu_soft_duty_cycle": GPU_DUTY_CYCLE,\n        "files": {\n            "preregistration": str(PREREG_PATH),\n            "risk_manifest": str(RISK_MANIFEST),\n            "candidate_checkpoint": str(branch_dir("B") / "branch_B_final.pt"),\n            "comparison": str(RUN_DIR / "EXP048_vs_EXP047B.csv"),\n            "cohort_comparison": str(cohort_comparison_path) if cohort_comparison_path else None,\n        },\n        "caveat": (\n            "Raw AP and equal-budget LOW_TP/HARD_FP recall are mandatory before final ADOPT_ULTRA_TAIL. "\n            "The inherited EXP047 evaluator persists AP5 and cohort scores but not exact raw AP."\n        ),\n    }\n    atomic_json(RESULT_PATH, result)\n    print(comparison.to_string(index=False))\n    print(json.dumps(result, indent=2))\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    subparsers = parser.add_subparsers(dest="command", required=True)\n    subparsers.add_parser("preregister")\n    subparsers.add_parser("inspect")\n    subparsers.add_parser("prepare-risk-cache")\n    smoke_parser = subparsers.add_parser("smoke")\n    smoke_parser.add_argument("--branch", required=True, choices=["A", "B"])\n    train_parser = subparsers.add_parser("train")\n    train_parser.add_argument("--branch", required=True, choices=["A", "B"])\n    evaluate_parser = subparsers.add_parser("evaluate")\n    evaluate_parser.add_argument("--branch", required=True, choices=["A", "B"])\n    subparsers.add_parser("finalize")\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    if args.command == "preregister":\n        preregister()\n    elif args.command == "inspect":\n        inspect_inputs()\n    elif args.command == "prepare-risk-cache":\n        prepare_risk_cache()\n    elif args.command == "smoke":\n        smoke(args.branch)\n    elif args.command == "train":\n        train(args.branch)\n    elif args.command == "evaluate":\n        evaluate(args.branch)\n    elif args.command == "finalize":\n        finalize()\n    else:\n        raise AssertionError(args.command)\n\n\nif __name__ == "__main__":\n    main()\n'
EXPECTED_EXP048_SHA = "4bcf3cb32421efa197d8711746fa4a936c0ed841e66a25364eaf03e291bd4f71"

source_sha = hashlib.sha256(RUNNER_SOURCE.encode("utf-8")).hexdigest()
assert source_sha == EXPECTED_EXP048_SHA
compile(RUNNER_SOURCE, str(TARGET), "exec")

if TARGET.exists():
    existing = hashlib.sha256(TARGET.read_bytes()).hexdigest()
    assert existing == EXPECTED_EXP048_SHA, (
        f"Existing EXP048 runner has different SHA: {existing}. "
        "Do not overwrite silently."
    )
else:
    TARGET.write_text(RUNNER_SOURCE, encoding="utf-8")

print("EXP047 SHA256:", actual_sha)
print("EXP048 runner SHA256:", EXPECTED_EXP048_SHA)
print("Installed:", TARGET)


AGENTS.md: D:\Projects\EPIC\AGENTS.md
# Project agent instructions

- Run all Python commands, notebooks, analyses, and tests for this workspace in the Conda environment `epic-eda`.
- On this Windows machine, prefer the explicit interpreter `C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe` when invoking Python non-interactively.
- Do not use the repository `.venv` or the system Python for project work unless the user explicitly requests it.

## Decision-making and scientific judgment

- Do not automatically agree with the user's proposed interpretation, experiment, architecture, or next step.
- Treat the user's preferred direction as a hypothesis to evaluate, not as the conclusion to defend.
- Independently compare the available alternatives using experimental evidence, expected information gain, methodological validity, cost, risk, and relevance to the primary objective.
- Recommend the path that is most logically supported by the evidence, even when it conflicts with the use

## Pre-register and inspect

These stages do not train and do not touch candidate validation.


In [2]:
env = os.environ.copy()
env["EPIC_GPU_DUTY_CYCLE"] = "0.90"

def run_stage(*args):
    cmd = [str(EXPECTED_PYTHON), "-u", str(TARGET), *args]
    print("RUN:", subprocess.list2cmdline(cmd), flush=True)
    p = subprocess.run(cmd, cwd=PROJECT_ROOT, env=env)
    if p.returncode != 0:
        raise RuntimeError((p.returncode, cmd))

run_stage("preregister")
run_stage("inspect")


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp048_ultra_tail.py preregister
RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp048_ultra_tail.py inspect


## Build the five-stratum cache

This reuses the existing corrected EXP045 train logits from EXP046B CV shards; it does **not**
run a new genome-wide backbone inference.


In [3]:
run_stage("prepare-risk-cache")

RUN_DIR = PROJECT_ROOT / "artifacts/experiments/CNN_EXP048_WIDTH512_ULTRA_TAIL_RISK_REFINEMENT/run_001"
manifest_path = RUN_DIR / "risk_strata_cache/risk_strata_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

import numpy as np
fractions = np.asarray(manifest["natural_fractions"], dtype=np.float64)
target = np.asarray(manifest["target_negative_mixture"], dtype=np.float64)
multipliers = np.asarray(manifest["candidate_negative_weight_multipliers"], dtype=np.float64)

assert len(fractions) == len(target) == len(multipliers) == 5
assert np.allclose(target, [0.065,0.065,0.145,0.175,0.55], atol=1e-12, rtol=0)
assert abs(target[:2].sum() - 0.13) < 1e-12
assert np.allclose(multipliers, target/fractions, rtol=1e-12, atol=1e-12)
assert manifest["rank_cut_counts"] == [
    round(manifest["total_train_negatives"] * x) for x in [0.001,0.01,0.05,0.15]
]

print("Five-stratum risk cache audit: PASS")
print("natural fractions:", fractions)
print("target masses:", target)
print("multipliers:", multipliers)


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp048_ultra_tail.py prepare-risk-cache
Five-stratum risk cache audit: PASS
natural fractions: [0.001 0.009 0.04  0.1   0.85 ]
target masses: [0.065 0.065 0.145 0.175 0.55 ]
multipliers: [64.99991928  7.22222382  3.62499987  1.74999998  0.64705883]


## Smoke: forward / backward / checkpoint restore

Candidate branch only.


In [4]:
run_stage("smoke", "--branch", "B")


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp048_ultra_tail.py smoke --branch B


## Train EXP048 candidate

+20k continuation steps from EXP045 @50k.

`EPIC_GPU_DUTY_CYCLE=0.90` inserts short idle periods after each optimizer update.
This lowers average duty/heat and leaves desktop headroom without changing batches, RNG,
optimizer state, gradients or LR schedule. It is **not** a strict Task Manager 90% ceiling.


In [5]:
run_stage("train", "--branch", "B")


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp048_ultra_tail.py train --branch B


## One final natural validation pass


In [6]:
run_stage("evaluate", "--branch", "B")


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp048_ultra_tail.py evaluate --branch B


## Compare against adopted EXP047-B

`finalize` does not rerun validation. It compares the just-written EXP048 metrics with the
already completed EXP047-B metrics.

Final `ADOPT_ULTRA_TAIL` remains pending until raw-AP and equal-budget LOW_TP/HARD_FP recall
are audited, because the inherited EXP047 evaluator did not persist exact raw AP.


In [7]:
run_stage("finalize")


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_cnn_exp048_ultra_tail.py finalize
